In [2]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.features import SET_A_TREE
from src.evaluate import evaluate

PROC = PROJECT_ROOT / "data" / "processed"
train = pd.read_parquet(PROC / "train.parquet")
val = pd.read_parquet(PROC / "val.parquet")   # test.parquet stays sealed
y_train, y_val = train["isFraud"], val["isFraud"]
print(train.shape, val.shape)

(2646363, 14) (85954, 14)


In [3]:
from sklearn.ensemble import RandomForestClassifier

forest_grid = [
    dict(max_depth=12, min_samples_leaf=20,  max_features=1, max_samples=0.3),  # 4A settings (reference)
    dict(max_depth=12, min_samples_leaf=20,  max_features=2, max_samples=0.2),
    dict(max_depth=12, min_samples_leaf=20,  max_features=3, max_samples=0.2),
    dict(max_depth=16, min_samples_leaf=20,  max_features=2, max_samples=0.2),
    dict(max_depth=16, min_samples_leaf=50,  max_features=2, max_samples=0.2),
    dict(max_depth=20, min_samples_leaf=50,  max_features=2, max_samples=0.2),
    dict(max_depth=12, min_samples_leaf=100, max_features=2, max_samples=0.2),
    dict(max_depth=16, min_samples_leaf=10,  max_features=3, max_samples=0.2),
]

forest_rows = []
for i, cfg in enumerate(forest_grid):
    t0 = time.time()
    m = RandomForestClassifier(n_estimators=60, n_jobs=-1, random_state=42, **cfg)
    m.fit(train[SET_A_TREE], y_train)
    r = evaluate(y_val, m.predict_proba(val[SET_A_TREE])[:, 1])
    forest_rows.append({"id": i, **cfg, "pr_auc_adj": r["pr_auc_adj"], "roc_auc": r["roc_auc"],
                        "recall@1%FPR": r["recall@1%FPR"], "seconds": round(time.time() - t0, 1)})
    print(f"forest #{i}  adjPR={r['pr_auc_adj']:.4f}  rec1%={r['recall@1%FPR']:.3f}  ({forest_rows[-1]['seconds']}s)")

forest_res = pd.DataFrame(forest_rows)
forest_res.to_csv(PROJECT_ROOT / "reports" / "tuning_forest.csv", index=False)
forest_res.sort_values("pr_auc_adj", ascending=False).round(4)

forest #0  adjPR=0.2544  rec1%=0.453  (37.7s)
forest #1  adjPR=0.2503  rec1%=0.445  (33.9s)
forest #2  adjPR=0.2442  rec1%=0.445  (43.0s)
forest #3  adjPR=0.2487  rec1%=0.445  (43.0s)
forest #4  adjPR=0.2416  rec1%=0.441  (40.6s)
forest #5  adjPR=0.2417  rec1%=0.441  (49.9s)
forest #6  adjPR=0.1595  rec1%=0.434  (35.5s)
forest #7  adjPR=0.2404  rec1%=0.451  (58.1s)


,id,max_depth,min_samples_leaf,max_features,max_samples,pr_auc_adj,roc_auc,recall@1%FPR,seconds
0,0,12,20,1,0.3,0.2544,0.8792,0.4525,37.7
1,1,12,20,2,0.2,0.2503,0.8791,0.4454,33.9
3,3,16,20,2,0.2,0.2487,0.8722,0.4454,43.0
2,2,12,20,3,0.2,0.2442,0.8781,0.4454,43.0
5,5,20,50,2,0.2,0.2417,0.8621,0.4407,49.9
4,4,16,50,2,0.2,0.2416,0.8759,0.4407,40.6
7,7,16,10,3,0.2,0.2404,0.8734,0.4509,58.1
6,6,12,100,2,0.2,0.1595,0.8800,0.4335,35.5


In [4]:
from xgboost import XGBClassifier

xgb_grid = [
    dict(max_depth=5,  learning_rate=0.10, n_estimators=300, min_child_weight=1),              # 4A settings (reference)
    dict(max_depth=4,  learning_rate=0.10, n_estimators=300, min_child_weight=1),
    dict(max_depth=6,  learning_rate=0.10, n_estimators=300, min_child_weight=1),
    dict(max_depth=6,  learning_rate=0.10, n_estimators=300, min_child_weight=0.2),
    dict(max_depth=8,  learning_rate=0.05, n_estimators=400, min_child_weight=1),
    dict(max_depth=8,  learning_rate=0.05, n_estimators=400, min_child_weight=3),
    dict(max_depth=6,  learning_rate=0.05, n_estimators=500, min_child_weight=1, reg_lambda=10),
    dict(max_depth=10, learning_rate=0.03, n_estimators=500, min_child_weight=3),
]

xgb_rows = []
for i, cfg in enumerate(xgb_grid):
    t0 = time.time()
    m = XGBClassifier(subsample=0.8, tree_method="hist", n_jobs=-1, random_state=42, **cfg)
    m.fit(train[SET_A_TREE], y_train)
    r = evaluate(y_val, m.predict_proba(val[SET_A_TREE])[:, 1])
    xgb_rows.append({"id": i, **cfg, "pr_auc_adj": r["pr_auc_adj"], "roc_auc": r["roc_auc"],
                     "recall@1%FPR": r["recall@1%FPR"], "seconds": round(time.time() - t0, 1)})
    print(f"xgb #{i}  adjPR={r['pr_auc_adj']:.4f}  rec1%={r['recall@1%FPR']:.3f}  ({xgb_rows[-1]['seconds']}s)")

xgb_res = pd.DataFrame(xgb_rows)
xgb_res.to_csv(PROJECT_ROOT / "reports" / "tuning_xgb.csv", index=False)
xgb_res.sort_values("pr_auc_adj", ascending=False).round(4)

xgb #0  adjPR=0.2434  rec1%=0.432  (41.9s)
xgb #1  adjPR=0.2464  rec1%=0.437  (39.7s)
xgb #2  adjPR=0.2383  rec1%=0.426  (40.3s)
xgb #3  adjPR=0.2366  rec1%=0.427  (67.0s)
xgb #4  adjPR=0.2333  rec1%=0.425  (228.4s)
xgb #5  adjPR=0.2380  rec1%=0.429  (166.2s)
xgb #6  adjPR=0.2449  rec1%=0.436  (178.6s)
xgb #7  adjPR=0.2359  rec1%=0.430  (236.1s)


,id,max_depth,learning_rate,n_estimators,min_child_weight,pr_auc_adj,roc_auc,recall@1%FPR,seconds,reg_lambda
1,1,4,0.10,300,1.0,0.2464,0.8782,0.4367,39.7,NaN
6,6,6,0.05,500,1.0,0.2449,0.8776,0.4359,178.6,10.0
0,0,5,0.10,300,1.0,0.2434,0.8733,0.4320,41.9,NaN
2,2,6,0.10,300,1.0,0.2383,0.8716,0.4256,40.3,NaN
5,5,8,0.05,400,3.0,0.2380,0.8729,0.4288,166.2,NaN
3,3,6,0.10,300,0.2,0.2366,0.8709,0.4272,67.0,NaN
7,7,10,0.03,500,3.0,0.2359,0.8700,0.4304,236.1,NaN
4,4,8,0.05,400,1.0,0.2333,0.8711,0.4248,228.4,NaN


In [5]:
best_f = forest_res.loc[forest_res["pr_auc_adj"].idxmax()]
best_x = xgb_res.loc[xgb_res["pr_auc_adj"].idxmax()]
f_cfg = {k: (int(best_f[k]) if k != "max_samples" else float(best_f[k]))
         for k in ["max_depth", "min_samples_leaf", "max_features", "max_samples"]}
x_cfg = {k: best_x[k] for k in ["max_depth", "learning_rate", "n_estimators", "min_child_weight"]}
x_cfg.update(max_depth=int(x_cfg["max_depth"]), n_estimators=int(x_cfg["n_estimators"]))
if "reg_lambda" in best_x and pd.notna(best_x["reg_lambda"]):
    x_cfg["reg_lambda"] = float(best_x["reg_lambda"])
print("best forest:", f_cfg, "\nbest xgb   :", x_cfg)

rows = []
for k in [1, 5, 20]:
    w = np.where(train["step"] > 408, float(k), 1.0)   # days 18-21 of training get weight k
    for name in ["forest", "xgboost"]:
        t0 = time.time()
        if name == "forest":
            m = RandomForestClassifier(n_estimators=60, n_jobs=-1, random_state=42, **f_cfg)
        else:
            m = XGBClassifier(subsample=0.8, tree_method="hist", n_jobs=-1, random_state=42, **x_cfg)
        m.fit(train[SET_A_TREE], y_train, sample_weight=w)
        r = evaluate(y_val, m.predict_proba(val[SET_A_TREE])[:, 1])
        rows.append({"model": name, "weight_on_days_18_21": k, "pr_auc_adj": r["pr_auc_adj"],
                     "recall@1%FPR": r["recall@1%FPR"], "seconds": round(time.time() - t0, 1)})
        print(name, k, round(r["pr_auc_adj"], 4))

pd.DataFrame(rows).round(4)

best forest: {'max_depth': 12, 'min_samples_leaf': 20, 'max_features': 1, 'max_samples': 0.3} 
best xgb   : {'max_depth': 4, 'learning_rate': np.float64(0.1), 'n_estimators': 300, 'min_child_weight': np.float64(1.0)}
forest 1 0.2588
xgboost 1 0.2464
forest 5 0.2563
xgboost 5 0.2394
forest 20 0.2318
xgboost 20 0.2096


,model,weight_on_days_18_21,pr_auc_adj,recall@1%FPR,seconds
0,forest,1,0.2588,0.4502,63.5
1,xgboost,1,0.2464,0.4367,42.1
2,forest,5,0.2563,0.4422,61.9
3,xgboost,5,0.2394,0.4328,48.6
4,forest,20,0.2318,0.4272,70.7
5,xgboost,20,0.2096,0.4201,37.5


In [6]:
final_cfg = dict(max_depth=4, learning_rate=0.1, n_estimators=300, min_child_weight=1)

rows = []
for seed in range(5):
    t0 = time.time()
    m = XGBClassifier(subsample=0.8, tree_method="hist", n_jobs=-1, random_state=seed, **final_cfg)
    m.fit(train[SET_A_TREE], y_train)
    r = evaluate(y_val, m.predict_proba(val[SET_A_TREE])[:, 1])
    rows.append({"seed": seed, "pr_auc_adj": r["pr_auc_adj"], "roc_auc": r["roc_auc"],
                 "recall@1%FPR": r["recall@1%FPR"], "seconds": round(time.time() - t0, 1)})
    print(f"seed {seed}  adjPR={r['pr_auc_adj']:.4f}  rec1%={r['recall@1%FPR']:.3f}", flush=True)

seeds = pd.DataFrame(rows)
print()
print(seeds[["pr_auc_adj", "roc_auc", "recall@1%FPR"]].agg(["mean", "std", "min", "max"]).round(4))

seed 0  adjPR=0.2471  rec1%=0.423
seed 1  adjPR=0.2483  rec1%=0.431
seed 2  adjPR=0.2057  rec1%=0.427
seed 3  adjPR=0.2349  rec1%=0.430
seed 4  adjPR=0.2471  rec1%=0.429

      pr_auc_adj  roc_auc  recall@1%FPR
mean      0.2366   0.8753        0.4280
std       0.0181   0.0005        0.0030
min       0.2057   0.8744        0.4233
max       0.2483   0.8758        0.4312


In [7]:
import joblib, json

final = XGBClassifier(subsample=0.8, tree_method="hist", n_jobs=-1, random_state=42, **final_cfg)
final.fit(train[SET_A_TREE], y_train)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(final, MODEL_DIR / "xgb_final.joblib")

r = evaluate(y_val, final.predict_proba(val[SET_A_TREE])[:, 1])
meta = {
    "model": "XGBClassifier",
    "features": SET_A_TREE,
    "params": {**final_cfg, "subsample": 0.8, "random_state": 42},
    "trained_on": "train split only (steps 1-504), TRANSFER and CASH_OUT only",
    "validation": {k: round(float(v), 4) for k, v in r.items()},
}
(MODEL_DIR / "xgb_final.json").write_text(json.dumps(meta, indent=2))

size_mb = (MODEL_DIR / "xgb_final.joblib").stat().st_size / 1024**2
print(f"saved xgb_final.joblib ({size_mb:.2f} MB)")
print(meta["validation"])

saved xgb_final.joblib (0.49 MB)
{'roc_auc': 0.8782, 'pr_auc_raw': 0.411, 'pr_auc_baseline': 0.0147, 'pr_auc_adj': 0.2464, 'recall@0.1%FPR': 0.2555, 'recall@1%FPR': 0.4367}
